# M06 — create_agent 與 Middleware

本 notebook 對應 `README.md`，逐格執行即可。

主線：把 M04 手刻的「工具迴圈」交給 `create_agent` 一行搞定，
再認識 `response_format`（結構化最終答案）與 Middleware（在流程裡插入行為），
最後揭穿底層就是一張 LangGraph 圖——通往第二冊。

## 1. 環境準備

跟前面每個模組一樣：把 `_shared` 加進路徑，載入 `.env`，取得供應商無關的模型。
`parents[1]` 從本模組資料夾往上兩層到課程根目錄，才找得到 `_shared/`。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 先定義工具（沿用 M04 的 @tool）

工具定義跟 M04 一模一樣，這裡沒有任何新東西。
我們準備兩個簡單工具：加法與乘法，讓問題需要「多步驟」才能算完。
`create_agent` 待會就靠這兩個工具自己跑迴圈。

In [ ]:
from langchain.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b


# Quick sanity check: a tool is still just a callable with a schema.
print(add.name, "/", multiply.name)
# Expected output: add / multiply

## 3. create_agent：一行取代 M04 的手刻迴圈

回想 M04：你得手動執行 `tool_calls`、把結果包成 `ToolMessage`、再 invoke 一次……
`create_agent` 把整個 think→act→observe 迴圈包進去了。

三件套：`model`、`tools`、`system_prompt`。
呼叫時傳 `{"messages": [...]}`，不是純字串。

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools=[add, multiply],
    system_prompt="你是計算助理，需要時就呼叫工具，算完用一句話回覆結果。",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "先算 3 加 5，再把結果乘以 2，等於多少?"}]}
)

# The final answer is the LAST message's content.
print(result["messages"][-1].content)
# Expected output (approx): "(3+5)*2 = 16" 之類的一句話答案

## 4. 印出完整 result["messages"] 看 Agent 的思考軌跡

`result["messages"]` 不只有最後答案，而是**整段對話軌跡**：
你的問題 → 模型決定呼叫工具的 AIMessage（帶 tool_calls）→ 工具結果 ToolMessage → ... → 收尾。
這正是 M04 你手刻的那個迴圈，現在自動跑完了。

In [ ]:
# Walk the whole trajectory so you can SEE the loop create_agent ran for you.
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__          # HumanMessage / AIMessage / ToolMessage
    print(f"[{i}] {msg_type}")
    if getattr(msg, "content", ""):
        print("    content:", msg.content)
    # AIMessages may carry tool_calls (the "act" decision).
    for call in getattr(msg, "tool_calls", []) or []:
        print("    tool_call:", call["name"], call["args"])

# Expected output: 依序看到 HumanMessage -> AIMessage(tool_calls=add) ->
# ToolMessage(8) -> AIMessage(tool_calls=multiply) -> ToolMessage(16) -> AIMessage(最終答案)

🧪 **練習 1**

把上面的問題換成一個需要**三步驟**的算式，例如
「(2+3) 乘以 4，再加 10」。重新跑第 3、4 格，觀察 `result["messages"]` 變長：
Agent 會自己多跑幾圈工具呼叫，你完全不用改迴圈程式碼。

## 5. response_format：讓最終答案變成 Pydantic 結構

預設最後一則訊息是自由文字。但下游程式常常要的是**結構化物件**
（呼應 M02 的 `with_structured_output`，這次套在整個 Agent 上）。
只要把一個 Pydantic 類別傳給 `response_format`，Agent 跑完就幫你整理成該結構。

In [ ]:
from pydantic import BaseModel, Field


class CalcAnswer(BaseModel):
    """Structured final answer for a calculation task."""

    result: int = Field(description="最終計算結果")
    steps: str = Field(description="用一句話說明計算過程")


structured_agent = create_agent(
    model,
    tools=[add, multiply],
    system_prompt="你是計算助理，需要時就呼叫工具。",
    response_format=CalcAnswer,            # 也可寫 ProviderStrategy(CalcAnswer)
)

structured_result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "先算 6 乘以 7，再加 8，等於多少?"}]}
)

# The parsed structured output lives under "structured_response".
answer = structured_result["structured_response"]
print(type(answer).__name__)               # CalcAnswer
print("result:", answer.result)            # Expected: 50
print("steps :", answer.steps)             # 一句話過程說明

如果你想明確指定「用供應商原生的結構化輸出策略」，可以這樣寫
（效果一樣，只是把策略講明）：

In [ ]:
from langchain.agents.structured_output import ProviderStrategy

# Equivalent to response_format=CalcAnswer, but the strategy is explicit.
structured_agent_v2 = create_agent(
    model,
    tools=[add, multiply],
    response_format=ProviderStrategy(CalcAnswer),
)
print("structured_agent_v2 ready")
# Expected output: structured_agent_v2 ready

## 6. Middleware：在 Agent 流程裡插一隻手

Middleware 是**疊在 Agent 迴圈外的一圈行為**：你不用重寫迴圈，
只要宣告「我要加這個行為」。最常見的兩個：

- `SummarizationMiddleware`：對話太長時自動摘要壓縮舊訊息，避免爆 context。
- `HumanInTheLoopMiddleware`：執行特定（危險）工具前先暫停，等人核准。

下面用最簡單的形式示範 `SummarizationMiddleware`。
（Middleware 的精確參數依版本而定；不確定時就用簡單形式，重點是理解概念。）

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

# Same agent as before, but now long conversations get auto-summarized.
agent_with_mw = create_agent(
    model,
    tools=[add, multiply],
    system_prompt="你是計算助理。",
    middleware=[SummarizationMiddleware(model=model)],
)

mw_result = agent_with_mw.invoke(
    {"messages": [{"role": "user", "content": "100 加 1 是多少?"}]}
)
print(mw_result["messages"][-1].content)
# Expected output: 一句話答案（如 "101"）。對短對話 middleware 不會做事；
# 它的價值要在多輪、長對話時才會被觸發。

🧪 **練習 2**

概念題（不一定要寫程式）：假設你的 Agent 有一個 `delete_file(path)` 工具，
你**不希望**它未經人類確認就刪檔。你會用哪一個 Middleware？
提示：答案是 `HumanInTheLoopMiddleware`——它能在危險工具執行前暫停等人核准。
想想看：這個「暫停 → 等人 → 再續跑」的能力，是不是已經很像第二冊的人介入流程了？

## 7. 底層就是一張 LangGraph 圖

為什麼輸入輸出都是 `{"messages": [...]}`？為什麼軌跡會一路累積？
因為 `create_agent` 回傳的不是黑盒子，而是一張**編譯好的 LangGraph 圖**，
`messages` 就是這張圖的「狀態（state）」。

我們可以直接把圖畫出來印證——這正是第二冊會大量用到的視覺化工具。

In [ ]:
# create_agent returns a compiled LangGraph graph; we can render it.
print(agent.get_graph().draw_ascii())
# Expected output: 一張 ASCII 流程圖，大致是
#   __start__ -> agent(model) -> tools -> agent -> ... -> __end__
# 你會看到 "agent"（呼叫模型）與 "tools"（執行工具）兩個節點之間來回的迴圈，
# 那就是 M04 你手刻、而 create_agent 自動跑的那個迴圈。

## 小結 & 下一步

你已經完成：

- 用 `create_agent(model, tools, system_prompt)` 一行取代 M04 的手刻迴圈。
- 用 `invoke({"messages": [...]})` 呼叫，並從 `result["messages"]` 讀出整段思考軌跡。
- 用 `response_format=Pydantic類別` 讓 Agent 的最終答案是結構化物件。
- 用 Middleware（`SummarizationMiddleware` / `HumanInTheLoopMiddleware`）在 Agent 流程裡插入行為。
- 親眼看到 `create_agent` 底層就是一張 LangGraph 圖。

**第一冊到此收尾。** 你已經把模型、prompt、parser、tool、retriever、agent
每一塊積木都摸過一遍。但 `create_agent` 是「別人幫你畫好的那張圖」。

當你需要它沒幫你做的事——**完全控制迴圈、加記憶、人介入後續、多 Agent 協作**——
就得自己畫圖。那就是 **第二冊 — LangGraph**：
從 `StateGraph`、`START`、`END` 開始，親手畫出第一張圖，
然後你會發現，`create_agent` 不過是其中一個特例。